In [1]:
"""
Flat vs Spherical "Earth" Concept-Frustration Simulation
========================================================

This FULL script runs end-to-end.

It implements YOUR NEW GEOMETRY + CONCEPTS + TASK:

SPHERE (scenario=1):
  u ~ Unif(S^2)
  d ~ Unif[0,d_max)
  p = (1-d) u

CYLINDER (scenario=0):
  (x_u,y_u) ~ Unif(unit disk)
  d ~ Unif[0,d_max)
  p = (x_u, y_u, -d)

Concepts (UPDATED; surface-parallel distances, depth-restricted):
  - We restrict depth to d in [0, 1] via d_max=1.0.
  - Cylinder (disk at z=-d): "parallel-to-surface" distance is straight-line distance IN THE PLANE z=-d:
        C1 = ||(x,y) - (0, +1)||_2
        C2 = ||(x,y) - (0, -1)||_2
    (equivalently distance to (0,±1,-d) along the disk; z cancels)
  - Sphere (sphere of radius r=1-d): "parallel-to-surface" distance is GEODESIC distance on that sphere:
        r = 1 - d
        North pole at depth d: N_d = (0, +r, 0)
        South pole at depth d: S_d = (0, -r, 0)
        C1 = r * arccos( (p·N_d) / r^2 ) = r * arccos(u_y)
        C2 = r * arccos( (p·S_d) / r^2 ) = r * arccos(-u_y)
  - C3 = d (unknown concept)

Task (unchanged, both):
  y = 1{ ||p - E||_2 < R_task }, where E=(1,0,0)

NEW ADDITION:
  - Identify "frustrated" SAE atoms (for the known pair C1,C2) using Fisher geometry:
        j is frustrated iff sign(Z_12) != sign(S_1j * S_2j) and S_1j*S_2j != 0
  - Predict C3 from projections of a (=X) onto:
        (i) all frustrated atoms
        (ii) a matched number of randomly chosen non-frustrated atoms
    using ridge regression, report test MSE and R^2.

PATCH (2026-02-10):
  - Matched-min sampling for C3 regression baseline:
      Use m = min(#frustrated, #non-frustrated) atoms on EACH side
      (prevents non-frust set from becoming empty when #frustrated > K/2).
  - Added bookkeeping fields:
      n_frust_atoms_full, n_nonfrust_pool, n_atoms_matched
  - Updated progress print to show nF(full/match).
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# 0) Geometry data generation (UPDATED)
# ============================================================

NORTH   = np.array([0.0,  1.0, 0.0], dtype=np.float32)
SOUTH   = np.array([0.0, -1.0, 0.0], dtype=np.float32)
EQUATOR = np.array([1.0,  0.0, 0.0], dtype=np.float32)  # your e=(1,0,0)


def sample_uniform_disk(n: int, rng: np.random.Generator):
    """Uniform samples over the unit disk via polar coordinates."""
    theta = rng.uniform(0.0, 2.0 * np.pi, size=n)
    rr = np.sqrt(rng.uniform(0.0, 1.0, size=n))
    x = (rr * np.cos(theta)).astype(np.float32)
    y = (rr * np.sin(theta)).astype(np.float32)
    return x, y


def sample_uniform_sphere_directions(n: int, rng: np.random.Generator):
    """Uniform direction samples on S^2 via normal->normalize."""
    v = rng.normal(size=(n, 3)).astype(np.float32)
    v /= (np.linalg.norm(v, axis=1, keepdims=True) + 1e-12)
    return v


def sample_depth(n: int, rng: np.random.Generator, *, d_max: float = 1.0) -> np.ndarray:
    """d ~ Unif[0, d_max).  ### CHANGED ### default d_max is now 1.0 (restrict depth to [0,1])."""
    return rng.uniform(0.0, float(d_max), size=n).astype(np.float32)


def generate_sphere_dig_concepts(n: int, seed: int, *, d_max: float = 1.0, R: float = 0.75):
    """
    Sphere setup:
      u ~ Unif(S^2), d ~ Unif[0,d_max), p=(1-d)u

    Concepts:  ### CHANGED ###
      Let r = 1 - d (sphere radius at depth d).
      Define "surface-parallel" distances as GEODESIC distances on the radius-r sphere to:
        N_d = (0, +r, 0), S_d = (0, -r, 0).
      Then for p=r*u:
        C1 = r * arccos( (p·N_d)/r^2 ) = r * arccos(u_y)
        C2 = r * arccos( (p·S_d)/r^2 ) = r * arccos(-u_y)
      C3 = d

    Task (unchanged):
      y = 1{ ||p - EQUATOR||_2 < R }
    """
    rng = np.random.default_rng(seed)
    u = sample_uniform_sphere_directions(n, rng)          # (n,3), ||u||=1
    d = sample_depth(n, rng, d_max=d_max)                 # (n,), now in [0,1)
    r = (1.0 - d).astype(np.float32)                      # (n,)  ### CHANGED ###
    P = r[:, None] * u                                    # (n,3)

    # ### CHANGED ### Geodesic distances on radius-r sphere:
    # Since (p·N_d)/r^2 = u_y and (p·S_d)/r^2 = -u_y
    uy = u[:, 1].astype(np.float32)
    uy = np.clip(uy, -1.0, 1.0)

    c1 = (r * np.arccos(uy)).astype(np.float32)
    c2 = (r * np.arccos(-uy)).astype(np.float32)
    c3 = d.astype(np.float32)

    dist_to_e = np.linalg.norm(P - EQUATOR[None, :], axis=1).astype(np.float32)
    y_bin = (dist_to_e < float(R)).astype(np.int64)

    C = np.stack([c1, c2, c3], axis=1).astype(np.float32)
    return C, y_bin


def generate_cylinder_dig_concepts(n: int, seed: int, *, d_max: float = 1.0, R: float = 0.75):
    """
    Cylinder setup:
      (x_u,y_u) ~ Unif(unit disk), d ~ Unif[0,d_max), p=(x_u,y_u,-d)

    Concepts:  ### CHANGED ###
      Define "surface-parallel" distances on the disk plane z=-d to points:
        N_d = (0, +1, -d), S_d = (0, -1, -d).
      The intrinsic (geodesic) distance on the plane equals the straight-line distance in (x,y):
        C1 = sqrt( (x-0)^2 + (y-1)^2 )
        C2 = sqrt( (x-0)^2 + (y+1)^2 )
      C3 = d

    Task (unchanged):
      y = 1{ ||p - EQUATOR||_2 < R }
    """
    rng = np.random.default_rng(seed)
    x, y = sample_uniform_disk(n, rng)
    d = sample_depth(n, rng, d_max=d_max)
    z = (-d).astype(np.float32)

    P = np.stack([x, y, z], axis=1).astype(np.float32)

    # ### CHANGED ### Planar distances on disk at fixed depth (z cancels):
    c1 = np.sqrt((x * x) + (y - 1.0) * (y - 1.0)).astype(np.float32)
    c2 = np.sqrt((x * x) + (y + 1.0) * (y + 1.0)).astype(np.float32)
    c3 = d.astype(np.float32)

    dist_to_e = np.linalg.norm(P - EQUATOR[None, :], axis=1).astype(np.float32)
    y_bin = (dist_to_e < float(R)).astype(np.int64)

    C = np.stack([c1, c2, c3], axis=1).astype(np.float32)
    return C, y_bin


def concepts_to_signal(C: np.ndarray, *, r: int = 64, sigma_x: float = 0.3, seed: int = 0):
    """
    X = C @ A^T + eps

    Treat X as the model input / activation proxy (a).
    """
    rng = np.random.default_rng(seed)
    n, k = C.shape
    A = rng.normal(size=(r, k)).astype(np.float32)
    eps = rng.normal(scale=sigma_x, size=(n, r)).astype(np.float32)
    X = (C @ A.T) + eps
    return X.astype(np.float32), A


def generate_geo_dataset(*, scenario: int, n: int, r: int, sigma_x: float, seed: int, d_max: float, R_task: float):
    """
    Return:
      X: (n,r) signal / activation proxy
      C: (n,3) concepts
      y: (n,) labels
      B: (3,3) sample covariance of C
    """
    if int(scenario) == 0:
        C, y = generate_cylinder_dig_concepts(n=n, seed=seed, d_max=d_max, R=R_task)
    elif int(scenario) == 1:
        C, y = generate_sphere_dig_concepts(n=n, seed=seed, d_max=d_max, R=R_task)
    else:
        raise ValueError("scenario must be 0 (cylinder) or 1 (sphere)")

    X, _A = concepts_to_signal(C, r=r, sigma_x=sigma_x, seed=seed + 12345)
    B = np.cov(C, rowvar=False, bias=False).astype(np.float32)
    return X, C, y, B


# ============================================================
# 1) Models (unchanged)
# ============================================================

class BlackBoxMLP(nn.Module):
    def __init__(self, r: int, hidden: int = 128):
        super().__init__()
        self.fc1 = nn.Linear(r, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, x, return_hidden: bool = False):
        z = self.fc1(x)
        h = self.act(z)
        logit = self.fc2(h).squeeze(-1)
        if return_hidden:
            return logit, z, h
        return logit


class LinearCBM(nn.Module):
    def __init__(self, xdim: int, k_known: int):
        super().__init__()
        self.concept = nn.Linear(xdim, k_known)
        self.task = nn.Linear(k_known, 1)

    def forward(self, x):
        c = self.concept(x)
        y = self.task(c).squeeze(-1)
        return c, y


class SparseAE(nn.Module):
    def __init__(self, xdim: int, K: int):
        super().__init__()
        self.W = nn.Linear(xdim, K, bias=True)
        self.D = nn.Parameter(torch.randn(K, xdim) * 0.02)

    def forward(self, x):
        s = torch.relu(self.W(x))
        x_hat = s @ self.D
        return s, x_hat


# ============================================================
# 2) Helpers (unchanged)
# ============================================================

@torch.no_grad()
def accuracy_from_logits(logits: torch.Tensor, y_true_np: np.ndarray) -> float:
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= 0.5).astype(np.int64)
    return float((preds == y_true_np).mean())


def train_test_split_indices(n: int, train_frac: float, rng: np.random.Generator):
    perm = rng.permutation(n)
    split = int(train_frac * n)
    return perm[:split], perm[split:]


# ============================================================
# 3) Fisher helpers (unchanged)
# ============================================================

@torch.no_grad()
def bb_predict_proba(bb: BlackBoxMLP, X: np.ndarray, *, device="cpu") -> np.ndarray:
    bb.eval()
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    logits = bb(Xt)
    return torch.sigmoid(logits).detach().cpu().numpy()


@torch.no_grad()
def compute_fisher_on_input_x(bb: BlackBoxMLP, X: np.ndarray, *, device="cpu", ridge=1e-6):
    """
    Fisher-like metric on the INPUT X for the 1-hidden-layer MLP:
      F = E[ s(x) * g(x) g(x)^T ]  where s=p(1-p) and g = d logit / d x.
    """
    bb.eval()
    Xt = torch.tensor(X, dtype=torch.float32, device=device)

    logits, z, _ = bb(Xt, return_hidden=True)
    p = torch.sigmoid(logits)
    s = p * (1 - p)
    m = (z > 0).float()

    W_H = bb.fc1.weight                 # (H, r)
    w_l = bb.fc2.weight.squeeze(0)      # (H,)

    U = m * w_l.unsqueeze(0)            # (n, H)
    G = U @ W_H                         # (n, r)

    F = (G.T @ (G * s.unsqueeze(1))) / float(X.shape[0])
    F = F + ridge * torch.eye(F.shape[0], device=device)
    return F.cpu().numpy()


# ============================================================
# 4) Training (unchanged)
# ============================================================

def train_bb_minibatch(X_tr, y_tr, X_te, y_te, *, hidden=128, epochs=30, batch_size=512, lr=1e-3, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    r = X_tr.shape[1]
    model = BlackBoxMLP(r, hidden=hidden).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)
    Xv = torch.tensor(X_te, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb, yb = Xt[b], yt[b]
            opt.zero_grad()
            loss = bce(model(xb), yb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        acc_te = accuracy_from_logits(model(Xv), y_te)
    return model, acc_te


def train_cbm_linear_minibatch(X_tr, Ck_tr, y_tr, X_te, Ck_te, y_te, *,
                              epochs=30, batch_size=512, lr=1e-3, lambda_c=1.0, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    xdim = X_tr.shape[1]
    k = Ck_tr.shape[1]
    model = LinearCBM(xdim, k).to(device)

    opt = optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    Ct = torch.tensor(Ck_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)

    Xv = torch.tensor(X_te, dtype=torch.float32, device=device)
    Cv = torch.tensor(Ck_te, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb, cb, yb = Xt[b], Ct[b], yt[b]
            opt.zero_grad()
            c_hat, logit = model(xb)
            loss = bce(logit, yb) + lambda_c * mse(c_hat, cb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        c_hat_te, logit_te = model(Xv)
        acc_te = accuracy_from_logits(logit_te, y_te)
        mse_te = float(((c_hat_te - Cv)**2).mean().cpu().item())

    return model, acc_te, mse_te


def train_sae_minibatch(X_tr, *, K=60, epochs=60, batch_size=512, lr=2e-3, l1=1e-3, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    xdim = X_tr.shape[1]
    model = SparseAE(xdim, K).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb = Xt[b]
            opt.zero_grad()
            s_code, xhat = model(xb)
            loss = ((xhat - xb)**2).mean() + l1 * s_code.abs().mean()
            loss.backward()
            opt.step()

    return model


# ============================================================
# 5) Geometry cosines (unchanged)
# ============================================================

def fisher_cosine_matrix(W: np.ndarray, D: np.ndarray, F: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Fsym = 0.5 * (F + F.T)
    WFW = np.einsum("ih,hk,ik->i", W, Fsym, W)
    DFD = np.einsum("jh,hk,jk->j", D, Fsym, D)
    Wn = np.sqrt(np.maximum(WFW, eps))
    Dn = np.sqrt(np.maximum(DFD, eps))
    num = W @ Fsym @ D.T
    return num / (Wn[:, None] * Dn[None, :] + eps)


def euclid_cosine_matrix(W: np.ndarray, D: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Wn = np.sqrt(np.sum(W * W, axis=1, keepdims=True) + eps)
    Dn = np.sqrt(np.sum(D * D, axis=1, keepdims=True) + eps)
    return (W @ D.T) / (Wn @ Dn.T)


def fisher_cosine_self(W: np.ndarray, F: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Fsym = 0.5 * (F + F.T)
    WFW = np.einsum("ih,hk,ik->i", W, Fsym, W)
    Wn = np.sqrt(np.maximum(WFW, eps))
    num = W @ Fsym @ W.T
    return num / (Wn[:, None] * Wn[None, :] + eps)


def euclid_cosine_self(W: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Wn = np.sqrt(np.sum(W * W, axis=1) + eps)
    num = W @ W.T
    return num / (Wn[:, None] * Wn[None, :] + eps)


# ============================================================
# 6) Pair frustration metric (unchanged)
# ============================================================

def pair_soft_frustration_metric(S: np.ndarray, Z: np.ndarray, *, eps: float = 1e-6) -> dict:
    k, K = S.shape
    if k < 2:
        return {
            "pair_soft_mean": 0.0, "pair_soft_max": 0.0, "pair_soft_nonzero_frac": 0.0,
            "pair_soft_pairs": 0, "pair_soft_eps": float(eps), "mean_abs_Z": 0.0,
            "pair_raw_mean": 0.0, "pair_raw_max": 0.0, "pair_raw_nonzero_frac": 0.0,
        }

    soft_scores, raw_scores = [], []
    soft_nonzero = 0
    raw_nonzero = 0
    total_pairs = 0

    for l in range(k):
        for r in range(l + 1, k):
            total_pairs += 1
            z = float(Z[l, r])
            if z == 0.0:
                soft_scores.append(0.0); raw_scores.append(0.0)
                continue

            prod = S[l, :] * S[r, :]
            zsign = np.sign(z)
            psign = np.sign(prod)

            contr_mask = (psign != 0) & (psign != zsign)
            if not np.any(contr_mask):
                soft_scores.append(0.0); raw_scores.append(0.0)
                continue

            best_raw = float(np.max(np.abs(prod[contr_mask])))
            raw_scores.append(best_raw)
            if best_raw > 0.0:
                raw_nonzero += 1

            best_soft = best_raw / (abs(z) + eps)
            soft_scores.append(best_soft)
            if best_soft > 0.0:
                soft_nonzero += 1

    soft_scores = np.asarray(soft_scores, dtype=float)
    raw_scores = np.asarray(raw_scores, dtype=float)

    tri = np.triu_indices(k, 1)
    mean_abs_Z = float(np.mean(np.abs(Z[tri])))

    return {
        "pair_soft_mean": float(soft_scores.mean()) if soft_scores.size else 0.0,
        "pair_soft_max": float(soft_scores.max()) if soft_scores.size else 0.0,
        "pair_soft_nonzero_frac": float(soft_nonzero / total_pairs) if total_pairs else 0.0,
        "pair_soft_pairs": int(total_pairs),
        "pair_soft_eps": float(eps),
        "mean_abs_Z": float(mean_abs_Z),
        "pair_raw_mean": float(raw_scores.mean()) if raw_scores.size else 0.0,
        "pair_raw_max": float(raw_scores.max()) if raw_scores.size else 0.0,
        "pair_raw_nonzero_frac": float(raw_nonzero / total_pairs) if total_pairs else 0.0,
    }


def frob_norm(A: np.ndarray) -> float:
    return float(np.sqrt(np.sum(A * A)))


def frob_abs_rel(A: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> dict:
    diff = A - B
    abs_f = frob_norm(diff)
    denom = frob_norm(A) + eps
    rel_f = abs_f / denom
    return {"frob_abs": float(abs_f), "frob_rel": float(rel_f)}


def metrics_from_S_trimmed(S: np.ndarray) -> dict:
    j_star = np.argmax(np.abs(S), axis=1)
    best_abs_signed = S[np.arange(S.shape[0]), j_star]
    frac_best_abs_negative = float((best_abs_signed < 0).mean())
    mean_best_abs_signed = float(best_abs_signed.mean())
    best_pos = S.max(axis=1)
    best_neg = S.min(axis=1)
    margin = best_pos - np.abs(best_neg)
    frac_margin_negative = float((margin < 0).mean())
    return {
        "frac_best_abs_negative": frac_best_abs_negative,
        "mean_best_abs_signed": mean_best_abs_signed,
        "frac_margin_negative": frac_margin_negative,
    }


def cov_matrix(X: np.ndarray) -> np.ndarray:
    return np.cov(X, rowvar=False, bias=False)


def corr_from_cov(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    d = np.sqrt(np.maximum(np.diag(C), eps))
    return C / (d[:, None] * d[None, :] + eps)


# ============================================================
# 6.5) NEW: C3 prediction from frustrated vs non-frustrated atoms
# ============================================================

def _select_frustrated_atoms_pair12(S: np.ndarray, Z: np.ndarray, *, tiny: float = 1e-12) -> np.ndarray:
    """
    For k_known=2, select atoms j such that:
      sign(Z_12) != sign(S_1j*S_2j) and S_1j*S_2j != 0
    """
    z = float(Z[0, 1])
    if z == 0.0:
        return np.array([], dtype=np.int64)

    prod = S[0, :] * S[1, :]
    mask_nz = np.abs(prod) > tiny
    zsign = np.sign(z)
    psign = np.sign(prod)

    contr = mask_nz & (psign != 0) & (psign != zsign)
    return np.where(contr)[0].astype(np.int64)


def _project_X_onto_atoms(X: np.ndarray, D: np.ndarray, idx: np.ndarray) -> np.ndarray:
    """
    Features Phi = X @ D_sel^T where D_sel are chosen SAE vectors (rows of D).
    """
    if idx.size == 0:
        return np.zeros((X.shape[0], 0), dtype=np.float32)
    return (X @ D[idx, :].T).astype(np.float32)


def _ridge_predict(Xtr: np.ndarray, ytr: np.ndarray, Xte: np.ndarray, *, lam: float = 1e-3):
    """
    Ridge regression with intercept, closed-form.

    minimize ||X b + b0 - y||^2 + lam ||b||^2
    """
    Xtr = np.asarray(Xtr, dtype=np.float64)
    Xte = np.asarray(Xte, dtype=np.float64)
    ytr = np.asarray(ytr, dtype=np.float64).reshape(-1)

    x_mu = Xtr.mean(axis=0, keepdims=True) if Xtr.shape[1] else np.zeros((1, 0), dtype=np.float64)
    y_mu = float(ytr.mean())

    Xc = Xtr - x_mu
    yc = ytr - y_mu

    d = Xc.shape[1]
    if d == 0:
        # predict constant mean
        yhat = np.full((Xte.shape[0],), y_mu, dtype=np.float32)
        return yhat, (y_mu, np.zeros((0,), dtype=np.float32))

    A = Xc.T @ Xc + lam * np.eye(d)
    b = np.linalg.solve(A, Xc.T @ yc)
    b0 = y_mu - float((x_mu @ b.reshape(-1, 1)).squeeze())
    yhat = (Xte @ b) + b0
    return yhat.astype(np.float32), (float(b0), b.astype(np.float32))


def _mse_r2(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    mse = float(np.mean((y_true - y_pred) ** 2))
    var = float(np.var(y_true))
    r2 = float(1.0 - mse / (var + 1e-12))
    return mse, r2


# ============================================================
# 7) One run (scenario in {0,1})
# ============================================================

def run_one(
    *,
    nu: int,            # scenario id (0=cylinder, 1=sphere)
    alpha: float,       # UNUSED (compat)
    omega: float,       # UNUSED (compat)
    seed: int,
    k_known: int,       # ignored; forced to 2
    device="cpu",
    sigma_x: float = 0.3,
    p_lo: float = 0.2,
    p_hi: float = 0.8,
    min_keep: int = 200,
    r: int = 64,
    n: int = 8000,
    K_sae: int = 60,
    pair_soft_eps: float = 1e-6,
    d_max: float = 1.0,    # ### CHANGED ### default now 1.0 (depth in [0,1])
    R_task: float = 0.75,
    ridge_lam_c3: float = 1e-3,   # NEW
):
    # Force concept dimensionality: (c1,c2 known; c3 unknown)
    k_total = 3
    k_known = 2

    X, C, y, B = generate_geo_dataset(
        scenario=int(nu),
        n=int(n),
        r=int(r),
        sigma_x=float(sigma_x),
        seed=int(seed),
        d_max=float(d_max),
        R_task=float(R_task),
    )

    B_known = np.array(B[:k_known, :k_known], dtype=float)

    rng = np.random.default_rng(seed)
    tr_idx, te_idx = train_test_split_indices(len(X), 0.75, rng)

    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    Ck_tr, Ck_te = C[tr_idx, :k_known], C[te_idx, :k_known]

    # True unknown concept
    C3_tr = C[tr_idx, 2].astype(np.float32)
    C3_te = C[te_idx, 2].astype(np.float32)

    # (1) Black-box
    bb, bb_acc = train_bb_minibatch(
        X_tr, y_tr, X_te, y_te,
        hidden=128, epochs=30, batch_size=512, lr=1e-3,
        seed=seed, device=device
    )

    # (2) Fisher on uncertain subset (band by predicted p)
    p_lo_use, p_hi_use = (p_lo, p_hi) if p_lo < p_hi else (p_hi, p_lo)
    p_tr = bb_predict_proba(bb, X_tr, device=device)
    keep = np.where((p_tr >= p_lo_use) & (p_tr <= p_hi_use))[0]
    if keep.size < min_keep:
        order = np.argsort(np.abs(p_tr - 0.5))
        keep = order[:min_keep]
    F = compute_fisher_on_input_x(bb, X_tr[keep], device=device, ridge=1e-6)

    # (3) CBM
    cbm, cbm_acc, cbm_mse = train_cbm_linear_minibatch(
        X_tr, Ck_tr, y_tr,
        X_te, Ck_te, y_te,
        epochs=30, batch_size=512, lr=1e-3,
        lambda_c=1.0, seed=seed, device=device
    )
    Wc = cbm.concept.weight.detach().cpu().numpy()

    with torch.no_grad():
        Xv = torch.tensor(X_te, dtype=torch.float32, device=device)
        c_hat_te, _ = cbm(Xv)
        C_hat_te = c_hat_te.detach().cpu().numpy()

    # (4) SAE
    sae = train_sae_minibatch(
        X_tr,
        K=K_sae, epochs=60, batch_size=512, lr=2e-3, l1=1e-3,
        seed=seed, device=device
    )
    D = sae.D.detach().cpu().numpy()  # (K, r)

    # (5) S matrices
    S_f = fisher_cosine_matrix(Wc, D, F)
    S_e = euclid_cosine_matrix(Wc, D)

    # (6) Z matrices
    Z_f = fisher_cosine_self(Wc, F)
    Z_e = euclid_cosine_self(Wc)

    # ============================================================
    # NEW: C3 prediction from frustrated atoms vs matched non-frustrated atoms
    # ============================================================

    # -------------------------------
    # PATCH: matched-min sampling
    # -------------------------------
    # OLD behavior: if n_frust > n_nonfrust, non_frust_idx becomes empty,
    # leading to constant-mean baseline (R^2 ~ 0) for "non-frust".
    #
    # NEW behavior: use m = min(n_frust, n_nonfrust) and sample m from each side,
    # so both regressions have the same number of features.
    frust_idx_full = _select_frustrated_atoms_pair12(S_f, Z_f, tiny=1e-12)
    n_frust_full = int(frust_idx_full.size)

    all_idx = np.arange(D.shape[0], dtype=np.int64)
    non_frust_pool = np.setdiff1d(all_idx, frust_idx_full, assume_unique=False)
    n_nonfrust_pool = int(non_frust_pool.size)

    rng_atoms = np.random.default_rng(seed + 99991)
    m_match = int(min(n_frust_full, n_nonfrust_pool))

    if m_match > 0:
        frust_idx = rng_atoms.choice(frust_idx_full, size=m_match, replace=False)
        non_frust_idx = rng_atoms.choice(non_frust_pool, size=m_match, replace=False)
    else:
        frust_idx = np.array([], dtype=np.int64)
        non_frust_idx = np.array([], dtype=np.int64)

    PhiF_tr = _project_X_onto_atoms(X_tr, D, frust_idx)
    PhiF_te = _project_X_onto_atoms(X_te, D, frust_idx)
    PhiN_tr = _project_X_onto_atoms(X_tr, D, non_frust_idx)
    PhiN_te = _project_X_onto_atoms(X_te, D, non_frust_idx)

    # Ridge predictions
    yhatF_te, _ = _ridge_predict(PhiF_tr, C3_tr, PhiF_te, lam=ridge_lam_c3)
    yhatN_te, _ = _ridge_predict(PhiN_tr, C3_tr, PhiN_te, lam=ridge_lam_c3)

    C3_mse_frust, C3_r2_frust = _mse_r2(C3_te, yhatF_te)
    C3_mse_nonfrust, C3_r2_nonfrust = _mse_r2(C3_te, yhatN_te)

    # (7) pair frustration metrics
    mf = metrics_from_S_trimmed(S_f)
    me = metrics_from_S_trimmed(S_e)
    pairF = pair_soft_frustration_metric(S_f, Z_f, eps=pair_soft_eps)
    pairE = pair_soft_frustration_metric(S_e, Z_e, eps=pair_soft_eps)

    # (8) geometry difference
    Sd = frob_abs_rel(S_f, S_e)

    # (9) faithfulness vs sample B_known (2x2 vs 2x2)
    Cov_hat = cov_matrix(C_hat_te)
    Corr_hat = corr_from_cov(Cov_hat)
    Corr_B = corr_from_cov(B_known)
    cov_diff = frob_abs_rel(Cov_hat, B_known)
    corr_diff = frob_abs_rel(Corr_hat, Corr_B)

    return {
        "scenario": int(nu),
        "seed": int(seed),
        "k_known": int(k_known),

        "n": int(n),
        "r": int(r),
        "sigma_x": float(sigma_x),
        "d_max": float(d_max),
        "R_task": float(R_task),

        "p_lo": float(p_lo_use),
        "p_hi": float(p_hi_use),
        "F_keep_n": int(len(keep)),

        "bb_acc": float(bb_acc),
        "cbm_acc": float(cbm_acc),
        "cbm_mse": float(cbm_mse),

        "F_frac_best_abs_negative": float(mf["frac_best_abs_negative"]),
        "F_mean_best_abs_signed": float(mf["mean_best_abs_signed"]),
        "F_frac_margin_negative": float(mf["frac_margin_negative"]),

        "E_frac_best_abs_negative": float(me["frac_best_abs_negative"]),
        "E_mean_best_abs_signed": float(me["mean_best_abs_signed"]),
        "E_frac_margin_negative": float(me["frac_margin_negative"]),

        "F_pair_raw_mean": float(pairF["pair_raw_mean"]),
        "F_pair_raw_max": float(pairF["pair_raw_max"]),
        "E_pair_raw_mean": float(pairE["pair_raw_mean"]),
        "E_pair_raw_max": float(pairE["pair_raw_max"]),

        "F_pair_soft_mean": float(pairF["pair_soft_mean"]),
        "F_pair_soft_max": float(pairF["pair_soft_max"]),
        "F_mean_abs_Z": float(pairF["mean_abs_Z"]),
        "E_pair_soft_mean": float(pairE["pair_soft_mean"]),
        "E_pair_soft_max": float(pairE["pair_soft_max"]),
        "E_mean_abs_Z": float(pairE["mean_abs_Z"]),
        "pair_soft_eps": float(pairF["pair_soft_eps"]),

        "S_frob_abs": float(Sd["frob_abs"]),
        "S_frob_rel": float(Sd["frob_rel"]),

        "Cov_frob_abs": float(cov_diff["frob_abs"]),
        "Cov_frob_rel": float(cov_diff["frob_rel"]),
        "Corr_frob_abs": float(corr_diff["frob_abs"]),
        "Corr_frob_rel": float(corr_diff["frob_rel"]),

        # NEW: C3 prediction from SAE projections (with matched-min sampling)
        "n_frust_atoms_full": int(n_frust_full),      # total frust atoms found
        "n_nonfrust_pool": int(n_nonfrust_pool),      # total non-frust available
        "n_atoms_matched": int(m_match),              # used on EACH side

        # Backwards-compatible key (now means "USED frust atoms", not "ALL frust atoms")
        "n_frust_atoms": int(m_match),

        "C3_ridge_lam": float(ridge_lam_c3),
        "C3_mse_frust_atoms": float(C3_mse_frust),
        "C3_r2_frust_atoms": float(C3_r2_frust),
        "C3_mse_nonfrust_atoms": float(C3_mse_nonfrust),
        "C3_r2_nonfrust_atoms": float(C3_r2_nonfrust),
    }


# ============================================================
# 8) Sweep runner
# ============================================================

def run_sweep(
    *,
    scenarios,
    seeds,
    device="cpu",
    sigma_x=0.3,
    p_lo=0.2,
    p_hi=0.8,
    min_keep=200,
    r=64,
    n=8000,
    K_sae=60,
    pair_soft_eps=1e-6,
    d_max=1.0,        # ### CHANGED ### default now 1.0
    R_task=0.75,
    ridge_lam_c3=1e-3,
):
    rows = []
    total = len(scenarios) * len(seeds)
    t = 0

    for sc in scenarios:
        for seed in seeds:
            t += 1
            out = run_one(
                nu=int(sc),
                alpha=+1.0,        # unused
                omega=0.0,         # unused
                seed=int(seed),
                k_known=2,
                device=device,
                sigma_x=sigma_x,
                p_lo=p_lo,
                p_hi=p_hi,
                min_keep=min_keep,
                r=r,
                n=n,
                K_sae=K_sae,
                pair_soft_eps=pair_soft_eps,
                d_max=d_max,
                R_task=R_task,
                ridge_lam_c3=ridge_lam_c3,
            )
            rows.append(out)

            sc_name = "cyl" if out["scenario"] == 0 else "sph"
            print(
                f"[{t:03d}/{total:03d}] {sc_name:3s} seed={out['seed']:2d} keep={out['F_keep_n']:4d} | "
                f"bb={out['bb_acc']:.3f} cbm={out['cbm_acc']:.3f} mse={out['cbm_mse']:.3f} | "
                f"Gamma_F(raw_mean)={out['F_pair_raw_mean']:.4f} | "
                f"nF(full/match)={out['n_frust_atoms_full']:2d}/{out['n_atoms_matched']:2d} "
                f"C3R2(F/N)={out['C3_r2_frust_atoms']:.3f}/{out['C3_r2_nonfrust_atoms']:.3f}"
            )

    return rows


# ============================================================
# 9) Simple summariser
# ============================================================

def summarize(rows):
    rows = list(rows)
    if not rows:
        print("No rows to summarise.")
        return

    def _stats(vals):
        vals = np.asarray(vals, dtype=float)
        return float(np.nanmean(vals)), float(np.nanstd(vals, ddof=1) if np.sum(np.isfinite(vals)) > 1 else 0.0)

    for sc in [0, 1]:
        subset = [r for r in rows if int(r["scenario"]) == sc]
        name = "cylinder dig" if sc == 0 else "sphere dig"
        if not subset:
            print(f"{name}: no runs")
            continue

        gf_mean, gf_sd = _stats([r["F_pair_raw_mean"] for r in subset])
        bb_mean, bb_sd = _stats([r["bb_acc"] for r in subset])
        cbm_mean, cbm_sd = _stats([r["cbm_acc"] for r in subset])

        # "n_frust_atoms" now means matched/used atoms (see patch).
        nF_mean, nF_sd = _stats([r["n_frust_atoms"] for r in subset])
        c3F_mean, c3F_sd = _stats([r["C3_r2_frust_atoms"] for r in subset])
        c3N_mean, c3N_sd = _stats([r["C3_r2_nonfrust_atoms"] for r in subset])

        # extra bookkeeping summaries (optional)
        nFfull_mean, nFfull_sd = _stats([r.get("n_frust_atoms_full", np.nan) for r in subset])
        nMatch_mean, nMatch_sd = _stats([r.get("n_atoms_matched", np.nan) for r in subset])

        print(f"\n=== {name} (n_runs={len(subset)}) ===")
        print(f"BB acc                  : {bb_mean:.3f} ± {bb_sd:.3f}")
        print(f"CBM acc                 : {cbm_mean:.3f} ± {cbm_sd:.3f}")
        print(f"Gamma_F (raw mean)      : {gf_mean:.4f} ± {gf_sd:.4f}")
        print(f"# frust atoms (FULL)     : {nFfull_mean:.2f} ± {nFfull_sd:.2f}")
        print(f"# atoms matched (USED)   : {nMatch_mean:.2f} ± {nMatch_sd:.2f}")
        print(f"C3 R^2 (frust atoms)     : {c3F_mean:.3f} ± {c3F_sd:.3f}")
        print(f"C3 R^2 (non-frust match) : {c3N_mean:.3f} ± {c3N_sd:.3f}")


# ============================================================
# 10) Main
# ============================================================

if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"

    seeds = np.arange(1, 51)
    scenarios = [0, 1]   # 0=cylinder, 1=sphere

    n = 8000
    r = 64
    sigma_x = 0.3

    p_lo, p_hi = 0.2, 0.8
    min_keep = 200

    K_sae = 60
    pair_soft_eps = 1e-6

    # Your setup params
    d_max = 1.0      # ### CHANGED ### restrict depth to [0,1]
    R_task = 0.75

    # C3-from-atoms regression strength
    ridge_lam_c3 = 1e-3

    rows = run_sweep(
        scenarios=scenarios,
        seeds=seeds,
        device=device,
        sigma_x=sigma_x,
        p_lo=p_lo,
        p_hi=p_hi,
        min_keep=min_keep,
        r=r,
        n=n,
        K_sae=K_sae,
        pair_soft_eps=pair_soft_eps,
        d_max=d_max,
        R_task=R_task,
        ridge_lam_c3=ridge_lam_c3,
    )

    print("\n\nSUMMARY")
    summarize(rows)

    # Optional: pack results dict for interactive use
    results = {
        "config": {
            "device": device,
            "seeds": seeds,
            "scenarios": scenarios,
            "scenario_meaning": {0: "cylinder dig", 1: "sphere dig"},
            "n": n,
            "r": r,
            "sigma_x": sigma_x,
            "p_band": (p_lo, p_hi),
            "min_keep": min_keep,
            "K_sae": K_sae,
            "pair_soft_eps": pair_soft_eps,
            "d_max": d_max,
            "R_task": R_task,
            "ridge_lam_c3": ridge_lam_c3,
            "binary_y_definition": "y=1 iff ||P-EQUATOR||_2 < R_task, where EQUATOR=(1,0,0)",
            "concepts": {
                "both": {
                    "c1": "surface-parallel distance to North pole at depth d (cylinder: planar on disk; sphere: geodesic on radius 1-d)",
                    "c2": "surface-parallel distance to South pole at depth d (cylinder: planar on disk; sphere: geodesic on radius 1-d)",
                    "c3": "d  (depth; unknown concept)",
                }
            },
            "sphere_geometry": "u~Unif(S^2), d~Unif[0,1), p=(1-d)u",
            "cylinder_geometry": "u~Unif(D^1), d~Unif[0,1), p=(x_u,y_u,-d)",
            "reference_points": {
                "NORTH": NORTH.tolist(),
                "SOUTH": SOUTH.tolist(),
                "EQUATOR": EQUATOR.tolist(),
            },
            "gamma_definitions": {"Gamma_F": "F_pair_raw_mean", "Gamma_E": "E_pair_raw_max"},
        },
        "rows": rows,
    }

/home/cbanerji/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


[001/100] cyl seed= 1 keep=1616 | bb=0.884 cbm=0.886 mse=0.034 | Gamma_F(raw_mean)=0.0155 | nF(full/match)= 7/ 7 C3R2(F/N)=0.401/0.940
[002/100] cyl seed= 2 keep=1620 | bb=0.887 cbm=0.884 mse=0.026 | Gamma_F(raw_mean)=0.0864 | nF(full/match)= 7/ 7 C3R2(F/N)=0.866/0.880
[003/100] cyl seed= 3 keep=1628 | bb=0.873 cbm=0.533 mse=0.030 | Gamma_F(raw_mean)=0.0025 | nF(full/match)= 2/ 2 C3R2(F/N)=0.265/0.577
[004/100] cyl seed= 4 keep=1585 | bb=0.873 cbm=0.840 mse=0.017 | Gamma_F(raw_mean)=0.3149 | nF(full/match)=16/16 C3R2(F/N)=0.950/0.978
[005/100] cyl seed= 5 keep=1665 | bb=0.875 cbm=0.879 mse=0.032 | Gamma_F(raw_mean)=0.0128 | nF(full/match)= 3/ 3 C3R2(F/N)=0.688/0.400
[006/100] cyl seed= 6 keep=1767 | bb=0.881 cbm=0.882 mse=0.031 | Gamma_F(raw_mean)=0.2611 | nF(full/match)=33/27 C3R2(F/N)=0.961/0.981
[007/100] cyl seed= 7 keep=1561 | bb=0.873 cbm=0.881 mse=0.015 | Gamma_F(raw_mean)=0.0000 | nF(full/match)= 0/ 0 C3R2(F/N)=-0.001/-0.001
[008/100] cyl seed= 8 keep=1615 | bb=0.883 cbm=0.883 

In [2]:
# Paired Wilcoxon comparison: cylinder vs sphere
# Metrics: bb_acc, cbm_acc, F_pair_raw_mean, E_pair_raw_mean
#
# Assumes you have `rows` in memory (list of dicts) produced by run_sweep().
# If you instead stored `results`, use: rows = results["rows"]

import numpy as np

# SciPy is the easiest route; if not installed, see the fallback below.
from scipy.stats import wilcoxon


def _paired_arrays(rows, metric, *, seed_key="seed"):
    """Return paired (cyl, sph) arrays matched by seed."""
    cyl = {int(r[seed_key]): float(r[metric]) for r in rows if int(r["scenario"]) == 0 and metric in r}
    sph = {int(r[seed_key]): float(r[metric]) for r in rows if int(r["scenario"]) == 1 and metric in r}

    common_seeds = sorted(set(cyl.keys()) & set(sph.keys()))
    x = np.array([cyl[s] for s in common_seeds], dtype=float)  # cylinder
    y = np.array([sph[s] for s in common_seeds], dtype=float)  # sphere
    return common_seeds, x, y


def paired_wilcoxon_report(rows, metrics=("bb_acc", "cbm_acc", "F_pair_raw_mean", "E_pair_raw_mean")):
    print("Paired Wilcoxon: sphere vs cylinder (paired by seed)")
    print("H1: median(sphere - cylinder) != 0\n")

    for m in metrics:
        seeds, x_cyl, y_sph = _paired_arrays(rows, m)
        if len(seeds) < 5:
            print(f"{m}: not enough paired seeds (n={len(seeds)})")
            continue

        diff = y_sph - x_cyl
        median_diff = float(np.median(diff))
        mean_cyl = float(np.mean(x_cyl))
        mean_sph = float(np.mean(y_sph))

        # Wilcoxon needs nonzero diffs; scipy handles 'zero_method'
        stat, p = wilcoxon(diff, zero_method="wilcox", alternative="two-sided")

        print(f"--- {m} ---")
        print(f"n (paired seeds)          : {len(seeds)}")
        print(f"mean cylinder / sphere    : {mean_cyl:.4f} / {mean_sph:.4f}")
        print(f"median(sphere - cylinder) : {median_diff:.6f}")
        print(f"Wilcoxon W                : {float(stat):.3f}")
        print(f"p-value                   : {float(p):.6g}\n")


# Run it
paired_wilcoxon_report(rows)

Paired Wilcoxon: sphere vs cylinder (paired by seed)
H1: median(sphere - cylinder) != 0

--- bb_acc ---
n (paired seeds)          : 50
mean cylinder / sphere    : 0.8756 / 0.8935
median(sphere - cylinder) : 0.018000
Wilcoxon W                : 3.000
p-value                   : 9.03763e-10

--- cbm_acc ---
n (paired seeds)          : 50
mean cylinder / sphere    : 0.8105 / 0.8236
median(sphere - cylinder) : 0.012500
Wilcoxon W                : 338.000
p-value                   : 0.00383703

--- F_pair_raw_mean ---
n (paired seeds)          : 50
mean cylinder / sphere    : 0.1264 / 0.1984
median(sphere - cylinder) : 0.055814
Wilcoxon W                : 352.000
p-value                   : 0.00523653

--- E_pair_raw_mean ---
n (paired seeds)          : 50
mean cylinder / sphere    : 0.0485 / 0.0447
median(sphere - cylinder) : -0.002197
Wilcoxon W                : 475.000
p-value                   : 0.118485



In [3]:
import pandas as pd

# Convert rows to DataFrame
df = pd.DataFrame(results["rows"])

# Export raw results
df.to_csv("results_globe_2_3_26_corrected_concepts.csv", index=False)
###i think no scale is better
print("Saved raw results to results_raw.csv")

Saved raw results to results_raw.csv
